In [1]:
import os
import pyspark

# Auto-detect Spark version → choose correct Kafka connector
spark_version = pyspark.__version__
print(f"Detected PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Using connector: {KAFKA_PACKAGE}")

# MUST be set BEFORE SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

Detected PySpark version: 4.0.0.dev2
Using connector: org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [3]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
from pyspark.sql.functions import col

batch_counter = {"n": 0}

def peek_raw(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id}: {df.count()} messages ---")
    df.select(
        "topic", "partition", "offset", "timestamp",
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),
    ).show(5, truncate=100)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    kafka_raw.writeStream
    .foreachBatch(peek_raw)
    .option("checkpointLocation", "/tmp/chk_lab4_peek")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

--- Batch 2: 3 messages ---
+------------+---------+------+-----------------------+----+----------------------------------------------------------------------------------------------------+
|       topic|partition|offset|              timestamp| key|                                                                                               value|
+------------+---------+------+-----------------------+----+----------------------------------------------------------------------------------------------------+
|transactions|        0|    51|2026-05-18 22:09:13.057|NULL|{"tx_id": "TX4561", "user_id": "u10", "amount": 3464.35, "store": "Gdansk", "category": "electron...|
|transactions|        0|    52|2026-05-18 22:09:14.058|NULL|{"tx_id": "TX5715", "user_id": "u08", "amount": 3357.9, "store": "Warsaw", "category": "clothing"...|
|transactions|        0|    53|2026-05-18 22:09:15.059|NULL|{"tx_id": "TX3782", "user_id": "u19", "amount": 3544.53, "store": "Warsaw", "category": "books", ...|


In [5]:
step1 = kafka_raw.select(
    col("offset"),
    col("partition"),
    col("value").cast("string").alias("raw_json"),
)

batch_counter["n"] = 0

def show_step1(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(3, truncate=120)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = step1.writeStream.foreachBatch(show_step1) \
    .option("checkpointLocation", "/tmp/chk_lab4_step1").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

step2 = kafka_raw.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
)

batch_counter["n"] = 0

def show_step2(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} — schema after from_json ---")
    df.printSchema()
    df.show(3, truncate=False)
    if batch_counter["n"] >= 1:
        raise Exception("stop")

q = step2.writeStream.foreachBatch(show_step2) \
    .option("checkpointLocation", "/tmp/chk_lab4_step2").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [7]:
from pyspark.sql.functions import to_timestamp

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss"))
)

print("Final schema:")
df.printSchema()

batch_counter["n"] = 0

def show_parsed(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = df.writeStream.foreachBatch(show_parsed) \
    .option("checkpointLocation", "/tmp/chk_lab4_parsed").start()
try:
    q.awaitTermination()
except:
    q.stop()

Final schema:
root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [8]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_window(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option("checkpointLocation", "/tmp/chk_lab4_windows")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [9]:
windowed_cat = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_cat(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} (complete mode — full table) ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "category", "tx_count", "total_amount",
        )
        .orderBy("from", "category")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed_cat.writeStream
    .outputMode("complete")   # ← KHÁC Task 4.1
    .foreachBatch(show_cat)
    .option("checkpointLocation", "/tmp/chk_lab4_windows_cat")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [10]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts")
    .outputMode("append")
    .start()
)
print("Alert stream started. Will stop after a few seconds for the lab.")

# Run for ~30 seconds then stop
import time
time.sleep(30)
alert_query.stop()
print("Alert query stopped.")

Alert stream started. Will stop after a few seconds for the lab.
Alert query stopped.


In [11]:
# Homework 1: Sliding window (2 minutes / 1-minute step) on the Kafka stream per store.

sliding_store = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "2 minutes", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_sliding(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} — sliding 2min / 1min step ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 4:
        raise Exception("stop")

q = (
    sliding_store.writeStream
    .outputMode("append")
    .foreachBatch(show_sliding)
    .option("checkpointLocation", "/tmp/chk_lab4_hw1")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [12]:
# Homework 2: Add a 'ratio' field = amount / 400.0 (approximate average from Lab 2) to alerts.

from pyspark.sql.functions import to_json, struct, lit

alerts_with_ratio = (
    df
    .filter(col("amount") > 3000)
    .withColumn("ratio", _round(col("amount") / 400.0, 2))
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                "ratio",                        # ← field mới
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)

# Demo locally trước khi ghi vào Kafka — để xem JSON đã có 'ratio' chưa
batch_counter["n"] = 0

def show_alerts(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n--- Batch {batch_id} — alerts JSON preview ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    alerts_with_ratio.writeStream
    .foreachBatch(show_alerts)
    .option("checkpointLocation", "/tmp/chk_lab4_hw2_preview")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

# Bây giờ ghi vào Kafka topic 'alerts'
alert_query2 = (
    alerts_with_ratio.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_hw2_kafka")
    .outputMode("append")
    .start()
)

import time
time.sleep(20)
alert_query2.stop()
print("HW2 alert stream (with ratio) finished.")

HW2 alert stream (with ratio) finished.


In [13]:
# Homework 3: What happens to the results when you stop the producer and wait 2 minutes? Why?

# ANSWER:
# 
# Symptoms when the producer is stopped:
#
# 1) New transactions stop arriving in the 'transactions' topic immediately.
#
# 2) For the FIRST ~30-60 seconds, Spark may still emit results for windows that
#    had pending data — because watermark advances based on event-time of messages
#    already received.
#
# 3) After ~2 minutes:
#    - No new event-time progression → watermark FREEZES at the last observed time.
#    - In 'append' mode: windows are NEVER closed (watermark doesn't advance), so
#      no new output is emitted. The query keeps running but produces nothing.
#    - In 'complete' mode: the same full table is re-emitted every micro-batch
#      with no new data — looks "stuck" on the last results.
#
# 4) Spark itself does NOT crash — it just waits patiently for new data.
#    When the producer restarts, processing resumes from where it left off
#    (thanks to checkpointLocation).
#
# WHY:
# Spark Structured Streaming uses EVENT TIME (the 'timestamp' field inside the
# transaction), not WALL-CLOCK time, to advance the watermark. Without new events,
# event time cannot progress. Windows close only when the watermark crosses their
# end time — so without new data, windows stay open indefinitely.
#
# This is by design: it lets Spark handle late-arriving data correctly regardless
# of when the data physically arrives.

print("Homework 3 answer is in the comments above.")

Homework 3 answer is in the comments above.
